# 014 — Figure S9 (multi-level atlas) — PUBLICATION figure, regenerate only

**This is the manuscript Figure S9** (full-atlas UMAP + epithelial PHATE multi-level mapping). Load-and-plot from the pre-computed atlas objects — **no re-analysis**. Regenerated at 1200 dpi.

**Input:** `scAtlas_annotated.h5ad` (full) + `pdac_epithelial_annotated.h5ad` (epithelial), both in `data/processed_data/pdac_atlas/`
**Output:** `.../PANC_atlas_validation/FigS9_atlas-multilevel_1200dpi.png` (+ .pdf)


In [ ]:
import os, numpy as np, scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

DATA_DIR   = "/storage/users/job37yv/Projects/PANC_cancer/data/processed_data/pdac_atlas"
OUTPUT_DIR = "/storage/users/job37yv/Projects/PANC_cancer/code/scripts_beta/PANC_atlas_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

adata     = sc.read_h5ad(f"{DATA_DIR}/scAtlas_annotated.h5ad")          # full atlas (726k)
adata_epi = sc.read_h5ad(f"{DATA_DIR}/pdac_epithelial_annotated.h5ad")  # epithelial (300k)
print("full:", adata.shape, "| epithelial:", adata_epi.shape)

# shared colours + helpers (from the analysis notebook)
PURPLE='#8E44AD'; GREEN='#2ECC71'; ORANGE='#F39C12'; RED='#E74C3C'; GREY='#BDC3C7'; BLUE='#3498DB'

def get_expr(ad, gene):
    x = ad[:, gene].X
    return x.toarray().flatten() if hasattr(x, 'toarray') else np.asarray(x).flatten()

def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

def panel_title(ax, letter, subtitle, fontsize_letter=20, fontsize_sub=11):
    ax.set_title(letter, fontsize=fontsize_letter, fontweight='bold', loc='left', pad=8)
    ax.text(0.5, 1.04, subtitle, transform=ax.transAxes, ha='center',
            fontsize=fontsize_sub, fontweight='semibold')


In [ ]:
ok=True
for name,a,cols,obsm in [("full",adata,['Clusters'],['X_umap']),
                         ("epi",adata_epi,['dominant_leiden','Leiden2_score','coexpr_binary','DiseaseState','Treatment','TreatmentType','phase'],['X_phate'])]:
    m=[c for c in cols if c not in a.obs.columns]; mm=[k for k in obsm if k not in a.obsm]
    if m or mm: print(f"{name} MISSING obs:{m} obsm:{mm}"); ok=False
print("OK — Fig S9 inputs present.") if ok else print("Re-run analysis notebook to add missing fields.")

In [ ]:
###########################################################################
# CELL: PUBLICATION ATLAS FIGURE v4 — bigger fonts, legends outside,
#       panel I self-contained (no states_8 dependency)
###########################################################################

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import numpy as np

fig = plt.figure(figsize=(21, 28), facecolor='white')
# hspace opened up so the below-panel legends have room
gs = GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.22,
              left=0.08, right=0.97, top=0.92, bottom=0.05)

PURPLE = '#8E44AD'; RED = '#E74C3C'; ORANGE = '#F39C12'
BLUE = '#3498DB'; GREEN = '#27AE60'; GREY = '#BDC3C7'

# --- HELPERS ---

def clean_ax(ax, xlabel='', ylabel=''):
    ax.set_xticks([]); ax.set_yticks([])
    if xlabel: ax.set_xlabel(xlabel, fontsize=15, labelpad=4, fontweight='semibold')
    if ylabel: ax.set_ylabel(ylabel, fontsize=15, labelpad=4, fontweight='semibold')
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)
        spine.set_color('#333333')

def to_np(mask):
    return mask.values if hasattr(mask, 'values') else mask

def panel_label(ax, letter, x=-0.06, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=28,
            fontweight='bold', va='top', ha='left', fontfamily='sans-serif')

# legends now default to BELOW the panel (bbox in axes fraction) so they
# sit outside the plot box and can be aligned manually
def make_legend(ax, handles, ncol=1, fontsize=11, bbox=(0.5, -0.15)):
    return ax.legend(handles=handles, fontsize=fontsize, loc='upper center',
                     bbox_to_anchor=bbox,
                     framealpha=0.95, edgecolor='#BBBBBB', fancybox=False,
                     ncol=ncol, handlelength=1.2, handleheight=1.0,
                     borderpad=0.5, columnspacing=0.9, labelspacing=0.4,
                     handletextpad=0.5)

# --- MASKS ---
normal_m = adata_epi.obs['DiseaseState'].isin(['Donor','Adjacent normal'])
met_m = adata_epi.obs['DiseaseState']=='Metastatic lesion'
coex = adata_epi.obs['coexpr_binary'] == 'CDK1+/CDKN1A+'

full_coords = adata.obsm['X_umap']
epi_emb = adata_epi.obsm['X_phate']

clusters_all = adata.obs['Clusters'].unique().tolist()
cluster_colors = {}
for c in clusters_all:
    cl = str(c).upper()
    if 'CYCLING' in cl and 'DUCTAL' in cl: cluster_colors[c] = '#9B59B6'
    elif 'DUCTAL' in cl: cluster_colors[c] = '#3498DB'
    elif 'FIBROBLAST' in cl: cluster_colors[c] = '#E67E22'
    elif 'ENDOTHELIAL' in cl: cluster_colors[c] = '#1ABC9C'
    elif any(x in cl for x in ['MACROPHAGE','MYELOID','MONOCYTE','DC']): cluster_colors[c] = '#C0392B'
    elif any(x in cl for x in ['TNK','T CELL','NK','B CELL','B_CELL','PLASMA']): cluster_colors[c] = '#F1C40F'
    elif 'CYCLING' in cl: cluster_colors[c] = '#AF7AC5'
    elif any(x in cl for x in ['ACINAR','ENDOCRINE']): cluster_colors[c] = '#7F8C8D'
    elif any(x in cl for x in ['MAST']): cluster_colors[c] = '#95A5A6'
    elif any(x in cl for x in ['STELLATE','PERICYTE']): cluster_colors[c] = '#D35400'
    else: cluster_colors[c] = '#BDC3C7'

epi_clusters = [c for c in clusters_all if 'DUCTAL' in str(c).upper()]
epi_mask_full = adata.obs['Clusters'].isin(epi_clusters)
non_epi_full = ~epi_mask_full

ds_cols = {'Donor':'#D5D8DC','Adjacent normal':'#AEB6BF',
           'Primary tumor':'#5DADE2','Metastatic lesion':'#E74C3C'}

L_COLS = {'Leiden0':GREEN,'Leiden1':RED,'Leiden2':PURPLE,'Leiden3':ORANGE}

handles_leiden = [mpatches.Patch(color=c, label=l) for c, l in
                  [(GREEN,'L0: Epithelial'),(RED,'L1: TGF-β/EMT'),
                   (PURPLE,'L2: Bottleneck'),(ORANGE,'L3: G2/M')]]

# =====================================================================
# ROW 1: FULL ATLAS (UMAP)
# =====================================================================

# A: Cell types
ax = fig.add_subplot(gs[0, 0])
panel_label(ax, 'A')
for cl in clusters_all:
    mask = adata.obs['Clusters'] == cl
    col = cluster_colors.get(cl, '#BDC3C7')
    ax.scatter(full_coords[mask.values,0], full_coords[mask.values,1],
               s=0.02, c=col, alpha=0.25, rasterized=True)
make_legend(ax, [
    mpatches.Patch(color='#3498DB', label='Ductal'),
    mpatches.Patch(color='#9B59B6', label='Cyc. ductal'),
    mpatches.Patch(color='#E67E22', label='Fibroblast'),
    mpatches.Patch(color='#C0392B', label='Myeloid'),
    mpatches.Patch(color='#F1C40F', label='Lymphoid'),
    mpatches.Patch(color='#1ABC9C', label='Endothelial'),
    mpatches.Patch(color='#D35400', label='Stellate'),
    mpatches.Patch(color='#7F8C8D', label='Other'),
], ncol=4, fontsize=11)
ax.set_title('Cell types', fontsize=17, fontweight='semibold', pad=6)
ax.text(0.02, 0.97, f'n = {adata.n_obs:,}', transform=ax.transAxes,
        fontsize=12, va='top', style='italic')
clean_ax(ax, 'UMAP 1', 'UMAP 2')

# B: Leiden signatures
ax = fig.add_subplot(gs[0, 1])
panel_label(ax, 'B')
ax.scatter(full_coords[non_epi_full.values,0], full_coords[non_epi_full.values,1],
           s=0.02, c='#ECECEC', alpha=0.15, rasterized=True)
if 'dominant_leiden' in adata.obs.columns:
    for k in ['Leiden0','Leiden1','Leiden3','Leiden2']:
        m = epi_mask_full & (adata.obs['dominant_leiden'] == k)
        if m.sum() == 0: continue
        ax.scatter(full_coords[m.values,0], full_coords[m.values,1],
                   s=0.04, c=L_COLS[k], alpha=0.35 if k=='Leiden0' else 0.5, rasterized=True)
make_legend(ax, handles_leiden, ncol=4, fontsize=12)
ax.set_title('Leiden signatures (epithelial)', fontsize=17, fontweight='semibold', pad=6)
clean_ax(ax, 'UMAP 1', 'UMAP 2')

# C: Bottleneck score
ax = fig.add_subplot(gs[0, 2])
panel_label(ax, 'C')
ax.scatter(full_coords[non_epi_full.values,0], full_coords[non_epi_full.values,1],
           s=0.02, c='#ECECEC', alpha=0.15, rasterized=True)
if 'Leiden2_score' in adata.obs.columns:
    epi_idx = np.where(epi_mask_full.values)[0]
    epi_order = epi_idx[np.argsort(adata.obs.iloc[epi_idx]['Leiden2_score'].values)]
    sc_c = ax.scatter(full_coords[epi_order,0], full_coords[epi_order,1],
                      c=adata.obs.iloc[epi_order]['Leiden2_score'].values,
                      s=0.04, cmap='magma', vmin=-0.15, vmax=0.45, alpha=0.5, rasterized=True)
    cb = plt.colorbar(sc_c, ax=ax, shrink=0.5, pad=0.01, aspect=25)
    cb.set_label('L2 score', fontsize=13, labelpad=3)
    cb.ax.tick_params(labelsize=12)
ax.set_title('Bottleneck score (epithelial)', fontsize=17, fontweight='semibold', pad=6)
clean_ax(ax, 'UMAP 1', 'UMAP 2')

# =====================================================================
# ROW 2: EPITHELIAL PHATE
# =====================================================================

# D: Clusters
ax = fig.add_subplot(gs[1, 0])
panel_label(ax, 'D')
for cl in adata_epi.obs['Clusters'].unique():
    m = adata_epi.obs['Clusters'] == cl
    col = cluster_colors.get(cl, '#BDC3C7')
    ax.scatter(epi_emb[m.values,0], epi_emb[m.values,1],
               s=0.06, c=col, alpha=0.3, rasterized=True)
make_legend(ax, [mpatches.Patch(color='#3498DB', label='Ductal'),
                  mpatches.Patch(color='#9B59B6', label='Cycling ductal')],
            ncol=2, fontsize=12)
ax.set_title('Epithelial clusters', fontsize=17, fontweight='semibold', pad=6)
ax.text(0.02, 0.97, f'n = {adata_epi.n_obs:,}', transform=ax.transAxes,
        fontsize=12, va='top', style='italic')
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# E: Disease state
ax = fig.add_subplot(gs[1, 1])
panel_label(ax, 'E')
for ds in ['Primary tumor','Donor','Adjacent normal','Metastatic lesion']:
    m = adata_epi.obs['DiseaseState'] == ds
    if m.sum() == 0: continue
    ax.scatter(epi_emb[m.values,0], epi_emb[m.values,1],
               s=0.06, c=ds_cols[ds], alpha=0.3, rasterized=True)
make_legend(ax, [mpatches.Patch(color='#5DADE2', label='Primary'),
                  mpatches.Patch(color='#E74C3C', label='Metastatic'),
                  mpatches.Patch(color='#AEB6BF', label='Normal')],
            ncol=3, fontsize=12)
ax.set_title('Disease state', fontsize=17, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# F: Treatment type
ax = fig.add_subplot(gs[1, 2])
panel_label(ax, 'F')
naive_m = adata_epi.obs['Treatment'] == 'Treatment naïve'
ax.scatter(epi_emb[naive_m.values,0], epi_emb[naive_m.values,1],
           s=0.04, c='#D5E8D4', alpha=0.15, rasterized=True)
norm_m2 = adata_epi.obs['DiseaseState'].isin(['Donor','Adjacent normal'])
ax.scatter(epi_emb[norm_m2.values,0], epi_emb[norm_m2.values,1],
           s=0.04, c='#ECECEC', alpha=0.15, rasterized=True)

treat_colors = {'Gemzar/Abraxane':'#9B59B6', 'FOLFIRINOX':'#E67E22',
                'CRT':'#2E86C1', 'RT & chemotherapy':'#17A589',
                'FOLFIRINOX, Gemzar/Abraxane':'#D4AC0D'}
for tt, col in treat_colors.items():
    m = adata_epi.obs['TreatmentType'] == tt
    if m.sum() < 10: continue
    ax.scatter(epi_emb[m.values,0], epi_emb[m.values,1],
               s=0.8, c=col, alpha=0.6, rasterized=True)
met_m2 = adata_epi.obs['DiseaseState'] == 'Metastatic lesion'
ax.scatter(epi_emb[met_m2.values,0], epi_emb[met_m2.values,1],
           s=0.06, c='#F1948A', alpha=0.2, rasterized=True)

make_legend(ax, [mpatches.Patch(color='#D5E8D4', label='Naïve'),
                  mpatches.Patch(color='#9B59B6', label='GEM/Abr.'),
                  mpatches.Patch(color='#E67E22', label='FOLFIR.'),
                  mpatches.Patch(color='#2E86C1', label='CRT'),
                  mpatches.Patch(color='#17A589', label='RT+chemo'),
                  mpatches.Patch(color='#F1948A', label='Metastatic')],
            ncol=3, fontsize=11)
ax.set_title('Treatment type', fontsize=17, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# =====================================================================
# ROW 3: SIGNATURES + AXIS
# =====================================================================

# G: Leiden signatures
ax = fig.add_subplot(gs[2, 0])
panel_label(ax, 'G')
for k in ['Leiden0','Leiden1','Leiden3','Leiden2']:
    m = adata_epi.obs['dominant_leiden'] == k
    ax.scatter(epi_emb[m.values,0], epi_emb[m.values,1],
               s=0.06, c=L_COLS[k], alpha=0.15 if k=='Leiden0' else 0.45, rasterized=True)
make_legend(ax, handles_leiden, ncol=4, fontsize=12)
ax.set_title('Leiden signatures', fontsize=17, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# H: Bottleneck score
ax = fig.add_subplot(gs[2, 1])
panel_label(ax, 'H')
order = np.argsort(adata_epi.obs['Leiden2_score'].values)
sc_h = ax.scatter(epi_emb[order,0], epi_emb[order,1],
                  c=adata_epi.obs['Leiden2_score'].values[order],
                  s=0.06, cmap='magma', vmin=-0.15, vmax=0.45, alpha=0.5, rasterized=True)
cb = plt.colorbar(sc_h, ax=ax, shrink=0.5, pad=0.01, aspect=25)
cb.set_label('L2 score', fontsize=13, labelpad=3)
cb.ax.tick_params(labelsize=12)
ax.set_title('Bottleneck score', fontsize=17, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# I: Axis states  -- self-contained masks (no states_8 dependency)
ax = fig.add_subplot(gs[2, 2])
panel_label(ax, 'I')
ax.scatter(epi_emb[:,0], epi_emb[:,1], s=0.02, c='#ECECEC', alpha=0.12, rasterized=True)

# Use states_8 if it exists in the notebook; otherwise recompute from expression
try:
    wee1_only    = to_np(states_8['C−A−W+']['mask'])
    cdk1_only_np = to_np(states_8['C+A−W−']['mask'])
    triple_np    = to_np(states_8['C+A+W+']['mask'])
except (NameError, KeyError):
    _cdk1   = np.asarray(get_expr(adata_epi, 'CDK1')).ravel()
    _cdkn1a = np.asarray(get_expr(adata_epi, 'CDKN1A')).ravel()
    _wee1   = np.asarray(get_expr(adata_epi, 'WEE1')).ravel()
    wee1_only    = (_cdk1 == 0) & (_cdkn1a == 0) & (_wee1 > 0)   # C−A−W+
    cdk1_only_np = (_cdk1 > 0)  & (_cdkn1a == 0) & (_wee1 == 0)   # C+A−W−
    triple_np    = (_cdk1 > 0)  & (_cdkn1a > 0)  & (_wee1 > 0)    # C+A+W+

ax.scatter(epi_emb[wee1_only,0], epi_emb[wee1_only,1],
           s=0.3, c=BLUE, alpha=0.25, rasterized=True)
ax.scatter(epi_emb[cdk1_only_np,0], epi_emb[cdk1_only_np,1],
           s=0.8, c=ORANGE, alpha=0.4, rasterized=True)
ax.scatter(epi_emb[coex.values,0], epi_emb[coex.values,1],
           s=1.5, c='black', alpha=0.7, rasterized=True, zorder=4)
ax.scatter(epi_emb[triple_np,0], epi_emb[triple_np,1],
           s=3, c=PURPLE, alpha=0.9, rasterized=True, zorder=5)

make_legend(ax, [
    mpatches.Patch(color=PURPLE, label=f'C⁺A⁺W⁺ ({int(np.sum(triple_np)):,})'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='black',
               markersize=6, label=f'CDK1⁺/CDKN1A⁺ ({int(coex.sum()):,})'),
    mpatches.Patch(color=ORANGE, label=f'CDK1⁺ only ({int(np.sum(cdk1_only_np)):,})'),
    mpatches.Patch(color=BLUE, label=f'WEE1⁺ only ({int(np.sum(wee1_only)):,})')],
    ncol=2, fontsize=11)
ax.set_title('CDK1–CDKN1A–WEE1 axis states', fontsize=15, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# =====================================================================
# ROW 4: TREATMENT ZOOMED
# =====================================================================
gem_m = adata_epi.obs['TreatmentType'].str.contains('Gem|Abraxane', case=False, na=False)
folf_m2 = adata_epi.obs['TreatmentType'] == 'FOLFIRINOX'

# J: Gemcitabine
ax = fig.add_subplot(gs[3, 0])
panel_label(ax, 'J')
ax.scatter(epi_emb[:,0], epi_emb[:,1], s=0.02, c='#ECECEC', alpha=0.08, rasterized=True)
gem_g1 = gem_m & (adata_epi.obs['phase']=='G1')
gem_cyc = gem_m & adata_epi.obs['phase'].isin(['S','G2M'])
gem_coex = gem_m & coex
ax.scatter(epi_emb[gem_g1.values,0], epi_emb[gem_g1.values,1],
           s=1, c='#D2B4DE', alpha=0.5, rasterized=True)
ax.scatter(epi_emb[gem_cyc.values,0], epi_emb[gem_cyc.values,1],
           s=2.5, c='#8E44AD', alpha=0.8, rasterized=True)
ax.scatter(epi_emb[gem_coex.values,0], epi_emb[gem_coex.values,1],
           s=25, c='#F1C40F', edgecolors='black', linewidth=0.5,
           alpha=1, rasterized=True, zorder=5, marker='*')
make_legend(ax, [
    mpatches.Patch(color='#D2B4DE', label=f'G1 ({gem_g1.sum():,})'),
    mpatches.Patch(color='#8E44AD', label=f'Cycling ({gem_cyc.sum():,})'),
    plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='#F1C40F',
               markeredgecolor='black', markersize=11,
               label=f'CDK1⁺/CDKN1A⁺ ({gem_coex.sum()})'),
], ncol=3, fontsize=11)
ax.set_title(f'Gemcitabine/Abraxane (n={gem_m.sum():,})', fontsize=15, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# K: FOLFIRINOX
ax = fig.add_subplot(gs[3, 1])
panel_label(ax, 'K')
ax.scatter(epi_emb[:,0], epi_emb[:,1], s=0.02, c='#ECECEC', alpha=0.08, rasterized=True)
folf_g1 = folf_m2 & (adata_epi.obs['phase']=='G1')
folf_cyc = folf_m2 & adata_epi.obs['phase'].isin(['S','G2M'])
folf_coex = folf_m2 & coex
ax.scatter(epi_emb[folf_g1.values,0], epi_emb[folf_g1.values,1],
           s=1, c='#FAD7A0', alpha=0.5, rasterized=True)
ax.scatter(epi_emb[folf_cyc.values,0], epi_emb[folf_cyc.values,1],
           s=2.5, c='#E67E22', alpha=0.8, rasterized=True)
ax.scatter(epi_emb[folf_coex.values,0], epi_emb[folf_coex.values,1],
           s=25, c='#F1C40F', edgecolors='black', linewidth=0.5,
           alpha=1, rasterized=True, zorder=5, marker='*')
make_legend(ax, [
    mpatches.Patch(color='#FAD7A0', label=f'G1 ({folf_g1.sum():,})'),
    mpatches.Patch(color='#E67E22', label=f'Cycling ({folf_cyc.sum():,})'),
    plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='#F1C40F',
               markeredgecolor='black', markersize=11,
               label=f'CDK1⁺/CDKN1A⁺ ({folf_coex.sum()})'),
], ncol=3, fontsize=11)
ax.set_title(f'FOLFIRINOX (n={folf_m2.sum():,})', fontsize=15, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# L: Metastatic + other treatments
ax = fig.add_subplot(gs[3, 2])
panel_label(ax, 'L')
ax.scatter(epi_emb[:,0], epi_emb[:,1], s=0.02, c='#ECECEC', alpha=0.08, rasterized=True)
crt_m = adata_epi.obs['TreatmentType'] == 'CRT'
rt_m = adata_epi.obs['TreatmentType'] == 'RT & chemotherapy'
met_coex = met_m & coex
ax.scatter(epi_emb[crt_m.values,0], epi_emb[crt_m.values,1],
           s=0.8, c='#2E86C1', alpha=0.35, rasterized=True)
ax.scatter(epi_emb[rt_m.values,0], epi_emb[rt_m.values,1],
           s=0.8, c='#17A589', alpha=0.35, rasterized=True)
ax.scatter(epi_emb[met_m.values,0], epi_emb[met_m.values,1],
           s=0.3, c='#F5B7B1', alpha=0.25, rasterized=True)
ax.scatter(epi_emb[met_coex.values,0], epi_emb[met_coex.values,1],
           s=3, c=RED, alpha=0.85, rasterized=True, zorder=4)

make_legend(ax, [
    mpatches.Patch(color='#2E86C1', label=f'CRT ({crt_m.sum():,})'),
    mpatches.Patch(color='#17A589', label=f'RT+chemo ({rt_m.sum():,})'),
    mpatches.Patch(color='#F5B7B1', label=f'Metastatic ({met_m.sum():,})'),
    mpatches.Patch(color=RED, label=f'Met. co-expr ({met_coex.sum():,})'),
], ncol=2, fontsize=11)
ax.set_title('CRT, RT+chemo & Metastatic', fontsize=15, fontweight='semibold', pad=6)
clean_ax(ax, 'PHATE 1', 'PHATE 2')

# =====================================================================
# Row bracket labels (left side) — y positions re-tuned for hspace=0.55
# =====================================================================
bracket_labels = [
    (0.84, 'Full atlas\n726,107 cells\n(UMAP)'),
    (0.60, 'Epithelial\n300,577 cells\n(PHATE)'),
    (0.37, 'Signatures\n& axis states'),
    (0.13, 'Treatment-\nspecific views'),
]
for y, label in bracket_labels:
    fig.text(0.008, y, label, fontsize=13, fontweight='bold',
             rotation=90, va='center', ha='center', color='#333333',
             fontfamily='sans-serif')

# Suptitle
fig.suptitle(
    'Multi-level PDAC atlas with CDK1–CDKN1A–WEE1 bottleneck annotations\n'
    '726,107 cells · 231 patients · 12 independent studies',
    fontsize=19, fontweight='bold', y=0.965, fontfamily='sans-serif')

# =====================================================================
# SAVE at 1200 DPI
# =====================================================================
plt.savefig(f"{OUTPUT_DIR}/FigS9_atlas-multilevel_1200dpi.png", dpi=1200, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(f"{OUTPUT_DIR}/FigS9_atlas-multilevel.pdf", bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Saved 1200 DPI PNG + vector PDF to {OUTPUT_DIR}/")